# ABPT Expanded Diagnostic Campaign

**Закрытие открытых вопросов: 29 кейсов, Phase 1, contradiction diagnostic**

Этот notebook прогоняет три ключевых эксперимента:

1. **Expanded Diagnostic** — все 29 кейсов (13 старых + 16 новых), классификация mature/template/flat, rescue rate
2. **Phase 1 Closure** — валидация `tail_retention_ratio` на short/medium/long профилях
3. **Contradiction Diagnostic** — почему `proof_by_contradiction` даёт delta=-1 (H5a/H5b/H5c)

---
Runtime: **GPU (T4 / A100)**  
Время: ~20–40 мин на все три эксперимента

## 0. GPU Check

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU not found — Runtime -> Change runtime type -> GPU')

## 1. Clone repo (feature branch)

In [ ]:
import os

REPO_URL = 'https://github.com/kharkilirov1/Anchor-engine.git'
BRANCH = 'claude/review-project-commits-IJRAf'
REPO_DIR = '/content/ABPT'

if os.path.exists(REPO_DIR):
    print('Repo exists -> fetch + checkout branch')
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f'Cloning branch {BRANCH}...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -5

## 2. Install dependencies

In [ ]:
%%time
!pip install -q --upgrade pip
!pip install -q torch torchvision accelerate einops scipy numpy pytest
!pip install -q "transformers @ git+https://github.com/huggingface/transformers.git@main"
print('Dependencies installed')

## 3. Verify: run tests

In [ ]:
!python -m pytest tests/test_expanded_cases.py -v

## 4. Current state

In [ ]:
import json
from pathlib import Path

s = json.loads(Path('research_state.json').read_text())
print(f"Phase: {s['current_phase']}/6")
print(f"Budget: {s['budget_remaining']}/{s['experiments_max']}")
print()

# Show known facts
kf = s.get('known_facts', {})
print(f"Crystallization zone: L{kf.get('crystallization_zone', '?')}")
print(f"Optimal profile: {kf.get('optimal_anchor_profile', '?')}")
print(f"tail_retention rho (medium): {kf.get('tail_retention_ratio_rho_medium', '?')}")
print(f"Flat rescue cases: {kf.get('n_flat_rescue_cases', '?')} (need 5+)")
print(f"Total cases: {kf.get('n_cases_current', '?')} -> now 29")
print()

# Show hypotheses
for h in s.get('open_hypotheses', []):
    icon = {'untested': '[ ]', 'success': '[+]', 'failed': '[-]', 'partial': '[~]'}.get(h['status'], '[?]')
    print(f"  {icon} {h['id']} (Phase {h['phase']}): {h['statement'][:60]}")

## 5. Expanded Diagnostic (29 cases, 3 clusters)

Main goal: get **5+ flat rescue cases** (currently 1).

New groups likely to be flat: medical (penicillin allergy), legal (GDPR), cross-domain (SQL+code).

In [ ]:
%%time
!python scripts/run_qwen_expanded_diagnostic.py \
    --model Qwen/Qwen3.5-4B \
    --anchor-profile medium

In [ ]:
# Show results
import json
from pathlib import Path

results_path = sorted(Path('archive').glob('*expanded_diagnostic*.json'),
                       key=lambda p: p.stat().st_mtime, reverse=True)
if not results_path:
    print('No results yet. Run cell above first.')
else:
    data = json.loads(results_path[0].read_text())
    print(f"File: {results_path[0].name}")
    print(f"Cases: {data['metadata']['n_cases_processed']}/{data['metadata']['n_cases_total']}")
    print(f"Correlation (tail_retention vs constraint): {data['correlation']['tail_retention_ratio_vs_base_constraint']}")
    print()
    
    print('CLUSTER SUMMARY:')
    print(f'{"Cluster":<12} {"Count":>6} {"Mean Base":>10} {"Rescued":>8} {"Rate":>8}')
    print('-' * 50)
    for cl in ('mature', 'template', 'flat'):
        s = data['cluster_summary'].get(cl, {})
        n = s.get('count', 0)
        mb = s.get('mean_base_constraint', '-')
        res = s.get('rescued', '-')
        rate = s.get('rescue_rate', '-')
        print(f'{cl:<12} {n:>6} {mb if isinstance(mb, str) else f"{mb:.3f}":>10} {res:>8} {rate if isinstance(rate, str) else f"{rate:.3f}":>8}')
    
    print()
    print('PER-CASE DETAILS:')
    print(f'{"Name":<40} {"Cluster":<10} {"Base":>5} {"Anchor":>7} {"Delta":>6}')
    print('-' * 70)
    for c in data['cases']:
        anchor_s = f"{c['anchor_constraint_score']:.0f}" if c.get('anchor_constraint_score') is not None else '-'
        tag = ' RESCUED' if c.get('rescued') else ''
        print(f"{c['name']:<40} {c['cluster']:<10} {c['base_constraint_score']:>5.0f} {anchor_s:>7} {c['delta']:>+6.0f}{tag}")

## 6. Phase 1 Closure (short / medium / long)

Criterion: `|rho(tail_retention_ratio, base_constraint)| > 0.4` on **ALL** profiles.

If `rho_spread < 0.3` across profiles -> thresholds are transferable.

In [ ]:
%%time
!python scripts/run_qwen_phase1_closure.py \
    --model Qwen/Qwen3.5-4B

In [ ]:
# Show Phase 1 verdict
import json
from pathlib import Path

p1_files = sorted(Path('archive').glob('*phase1_closure*.json'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
if not p1_files:
    print('No Phase 1 results. Run cell above.')
else:
    p1 = json.loads(p1_files[0].read_text())
    print(f"VERDICT: {p1['verdict']}")
    print(f"rho spread: {p1.get('rho_spread', 'N/A')}")
    print(f"Transferable: {p1.get('transferable', 'N/A')}")
    print()
    print(f'{"Profile":<10} {"rho":>8} {"Cases":>7} {"Status":>8}')
    print('-' * 35)
    for profile, info in p1['comparison'].items():
        rho = info.get('rho')
        rho_s = f'{rho:.4f}' if rho is not None else 'N/A'
        status = 'PASS' if info.get('passed') else 'FAIL'
        print(f'{profile:<10} {rho_s:>8} {info["n_cases"]:>7} {status:>8}')

## 7. Contradiction Diagnostic (proof_by_contradiction failure)

Tests three hypotheses:
- **H5a**: Carryover peak in handoff zone (L24-31) -> conflict
- **H5b**: Template cluster misclassification
- **H5c**: Base generation already correct, anchor breaks it

Also compares proof vs induction vs other procedure groups.

In [ ]:
%%time
!python scripts/run_qwen_contradiction_diagnostic.py \
    --model Qwen/Qwen3.5-4B

In [ ]:
# Show contradiction diagnostic
import json
from pathlib import Path

cd_files = sorted(Path('archive').glob('*contradiction_diagnostic*.json'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
if not cd_files:
    print('No contradiction diagnostic. Run cell above.')
else:
    cd = json.loads(cd_files[0].read_text())
    hyp = cd['hypotheses']
    
    print('HYPOTHESIS RESULTS:')
    print()
    
    h5a = hyp['H5a_carryover_in_handoff']
    print(f"  H5a (carryover in handoff zone):")
    print(f"    proof:  {h5a['proof_in_handoff']}/{h5a['proof_total']} in handoff")
    print(f"    other:  {h5a['other_in_handoff']}/{h5a['other_total']} in handoff")
    if h5a['proof_in_handoff'] > h5a['other_in_handoff']:
        print(f"    -> SUPPORTS H5a: proof carryover disproportionately in handoff zone")
    else:
        print(f"    -> DOES NOT SUPPORT H5a")
    
    print()
    h5b = hyp['H5b_delta_comparison']
    print(f"  H5b (delta comparison):")
    print(f"    proof mean delta:  {h5b['proof_mean_delta']:+.3f}" if h5b['proof_mean_delta'] is not None else "    proof: N/A")
    print(f"    other mean delta:  {h5b['other_mean_delta']:+.3f}" if h5b['other_mean_delta'] is not None else "    other: N/A")
    
    print()
    h5c = hyp['H5c_base_already_correct']
    print(f"  H5c (base already correct):")
    print(f"    proof base correct: {h5c['proof_base_correct']}/{h5c['proof_total']}")
    if h5c['proof_base_correct'] > 0:
        print(f"    -> SUPPORTS H5c: anchor BREAKS correct base generation")
    
    print()
    print('PER-CASE:')
    print(f'{"Name":<40} {"Group":<15} {"Carry@L":>8} {"Handoff":>8} {"Base":>5} {"Anchor":>7} {"Delta":>6}')
    print('-' * 90)
    for c in cd['cases']:
        hoff = 'YES' if c.get('carryover_in_handoff_zone') else 'no'
        print(f"{c['name']:<40} {c['group_label']:<15} L{c.get('carryover_peak_layer', '?'):>5} {hoff:>8} {c['base_score']:>5.0f} {c['anchor_score']:>7.0f} {c['delta']:>+6.0f}")

## 8. Summary & Next Steps

In [ ]:
print('='*60)
print('CAMPAIGN SUMMARY')
print('='*60)
print()

# Collect results
import json
from pathlib import Path

archive = Path('archive')

# Expanded diagnostic
exp_files = sorted(archive.glob('*expanded_diagnostic*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if exp_files:
    exp = json.loads(exp_files[0].read_text())
    flat_summary = exp['cluster_summary'].get('flat', {})
    n_flat = flat_summary.get('count', 0)
    n_rescued = flat_summary.get('rescued', 0)
    rescue_rate = flat_summary.get('rescue_rate', 0)
    rho = exp['correlation'].get('tail_retention_ratio_vs_base_constraint')
    print(f'[Expanded] Cases: {exp["metadata"]["n_cases_processed"]}')
    print(f'[Expanded] Flat cases: {n_flat}, rescued: {n_rescued}, rate: {rescue_rate}')
    print(f'[Expanded] tail_retention rho: {rho}')
    print(f'[Expanded] Flat rescue target (>=5): {"MET" if n_rescued >= 5 else "NOT MET"}')
else:
    print('[Expanded] Not run yet')

print()

# Phase 1
p1_files = sorted(archive.glob('*phase1_closure*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if p1_files:
    p1 = json.loads(p1_files[0].read_text())
    print(f'[Phase 1] Verdict: {p1["verdict"]}')
    for profile, info in p1['comparison'].items():
        rho = info.get('rho')
        print(f'  {profile}: rho={rho:.4f} {"PASS" if info["passed"] else "FAIL"}' if rho else f'  {profile}: N/A')
    print(f'[Phase 1] Transferable: {p1.get("transferable", "?")}')    
else:
    print('[Phase 1] Not run yet')

print()

# Contradiction
cd_files = sorted(archive.glob('*contradiction_diagnostic*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if cd_files:
    cd = json.loads(cd_files[0].read_text())
    h = cd['hypotheses']
    print(f'[Contradiction] H5a (handoff): proof {h["H5a_carryover_in_handoff"]["proof_in_handoff"]}/{h["H5a_carryover_in_handoff"]["proof_total"]} vs other {h["H5a_carryover_in_handoff"]["other_in_handoff"]}/{h["H5a_carryover_in_handoff"]["other_total"]}')
    print(f'[Contradiction] H5b (delta): proof={h["H5b_delta_comparison"]["proof_mean_delta"]}, other={h["H5b_delta_comparison"]["other_mean_delta"]}')
    print(f'[Contradiction] H5c (base ok): {h["H5c_base_already_correct"]["proof_base_correct"]}/{h["H5c_base_already_correct"]["proof_total"]}')
else:
    print('[Contradiction] Not run yet')

print()
print('='*60)
print('NEXT STEPS:')
print('  - If flat rescue >= 5: Phase 4 CLOSED')
print('  - If Phase 1 CONFIRMED: proceed to Phase 2 (group routing)')
print('  - If H5a supported: add template_verify route for handoff-zone anchors')
print('  - If rho transferable: auto-calibration approach validated')

## 9. Download results

In [ ]:
import shutil, os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
out_name = f'abpt_expanded_campaign_{timestamp}'
out_dir = f'/tmp/{out_name}'
os.makedirs(out_dir, exist_ok=True)

# Copy results
if os.path.exists('archive'):
    shutil.copytree('archive', f'{out_dir}/archive', dirs_exist_ok=True)
for f in ['playbook.md', 'research_state.json']:
    if os.path.exists(f):
        shutil.copy(f, out_dir)
if os.path.exists('docs/research'):
    shutil.copytree('docs/research', f'{out_dir}/docs_research', dirs_exist_ok=True)

shutil.make_archive(f'/tmp/{out_name}', 'zip', '/tmp', out_name)

try:
    from google.colab import files
    files.download(f'/tmp/{out_name}.zip')
    print(f'Downloaded: {out_name}.zip')
except ImportError:
    print(f'Saved: /tmp/{out_name}.zip')

---
## Cheatsheet

| Script | What it does |
|--------|-------------|
| `run_qwen_expanded_diagnostic.py` | All 29 cases, 3 clusters, rescue stats |
| `run_qwen_phase1_closure.py` | Validate tail_retention on short/medium/long |
| `run_qwen_contradiction_diagnostic.py` | H5a/H5b/H5c for proof failure |
| `run_qwen_phase_probe.py` | Single-profile phase probe |
| `run_qwen_per_case_diagnostic_v2.py` | Per-case detail |

**Key thresholds:**
- `mature_r1_threshold = 0.65` (confident crystallization)
- `template_delta_threshold = 0.08` (pattern/shortcut)
- `tail_retention_ratio rho > 0.4` = Phase 1 success
- `flat rescue cases >= 5` = Phase 4 closure

**New groups (likely flat):** penicillin allergy, GDPR, SQL FK, TypeScript null safety